# Natural-Language-to-CLI Command Generation for Network Management

**Mistral-7B-Instruct-v0.2 · Apple MLX · LoRA · Network management**

This notebook presents a proof-of-concept for translating natural-language requests into structured `br-t9s.cli extract ...` commands. It covers **command generation only**, not request classification or execution against a live network.

The repository includes the historical MLX LoRA implementation, a saved adapter (`adapters.npz`), and the original training, validation, and test examples. The three command outputs presented later are **recorded results from the development notebook**, not results from a new execution or a quantitative held-out evaluation.

## 1. Supported operations

| Operation | Example command |
| --- | --- |
| Extract an element's configuration | `br-t9s.cli extract config <element> <directory>` |
| Extract a network graph | `br-t9s.cli extract graph` |
| Extract infrastructure information | `br-t9s.cli extract infra` |
| Extract an element's logs | `br-t9s.cli extract logs <element>` |

The prompt also specifies command variants involving `--config`, `--namespace`, and `--verbose`. It is a command-generation experiment for a defined CLI grammar, **not** a general natural-language shell interface. Generated commands are treated as text and are not executed.

## 2. Environment and model configuration

This project uses the **legacy MLX LoRA implementation for Apple Silicon/macOS**, included in [`scripts/`](scripts/). The trained adapter follows the old `.npz` weight format and may not be compatible with current `mlx-lm` adapter-directory APIs. The scripts are adapted from Apple's MLX examples; see the repository's attribution notes.

The model weights are **not** included. At first use, the script attempts to load the specified MLX-compatible model from a local path or download it from Hugging Face. Start Jupyter **from the repository root** so `scripts/lora.py`, `data/`, and `adapters.npz` resolve correctly.

Exact dependency versions from the original runtime were not captured, and inference has not been rerun in the environment used to prepare this repository.

In [ ]:
from pathlib import Path
import re
import subprocess
import sys

MODEL_PATH = "mlx-community/Mistral-7B-Instruct-v0.2-4bit"
ADAPTER_PATH = Path("adapters.npz")
LORA_SCRIPT = Path("scripts/lora.py")
MAX_TOKENS = 140

## 3. Command specification

The prompt describes the allowed command grammar, arguments, and flags. It defines four operation families and remains close to the development prompt. The cleaned explanatory introduction is not character-for-character identical to the training examples; recorded outputs are therefore shown separately rather than attributed to a fresh run of this version.

In [ ]:
COMMAND_SPECIFICATION = """Translate the user request into exactly one supported CLI command. The four supported operations are configuration extraction, network graph extraction, infrastructure extraction, and log extraction. Preserve any relevant element names, paths, namespaces, or other arguments from the request. If multiple options are requested, include the applicable flags in the generated command. Return the CLI command only, without commentary or formatting.

Base command 1: If the user requests to extract the config out of an element into a directory in input and no other options are mentioned, detect the <element> which can be a random string, and that <directory> which can be a random string and add them in the output like this:
"br-t9s.cli extract config <element> <directory>"

Option of base command 1: If the user requests to extract the config out of an element into a directory in input and a config file is mentioned to perform operations,detect that <element> which can be a random string, that <directory> which can be a random string and <config> file for operations which can be a random string and add them in the output like this:(default value for <config> file is "config.json")
"br-t9s.cli extract config --config <element> <directory> <config>"

Option of base command 1: If the user requests to extract the config out of an element into a directory in input and a namespace is mentioned to perform operations, you should detect that <element> which can be a random string, that <directory> which can be a random string and the <namespace> for operations which can be a random string and add them in the output like this:(default value for <namespace> is "trirematics")
"br-t9s.cli extract config --namespace <element> <directory> <namespace>"

Option of base command 1: If the user requests to extract the config out of an element into a directory in input and also wants to debug the output of the command, you should detect that <element> which can be a random string, that <directory> which can be a random string and add "--verbose" argument besides them in the output like this:
"br-t9s.cli extract config --verbose <element> <directory>"



Base command 2: If user requests the network graph and no other options are mentioned, the output is like this:
"br-t9s.cli extract graph"

Option of base command 2: If the user requests to extract the network graph in input and a <config> file is mentioned to perform operations, you should detect that <config> file for operations which can be a random string and add them to the output like this:(default value for <config> file is "config.json")
"br-t9s.cli extract graph --config <config>"

Option of base command 2: If the user requests to extract the network graph in input and a namespace is mentioned to perform operations, you should detect that <namespace> which can be a random string and add them to the output like this:(default value for <namespace> is "trirematics")
"br-t9s.cli extract graph --namespace <namespace>"

Option of base command 2: If the user requests to extract the network graph in input and also want to debug the output, add argument "--verbose" besides them to the output like this:
"br-t9s.cli extract graph --verbose"


Base command 3: If the user wants the infrastructure data and no other options are mentioned, the output is like this:
"br-t9s.cli extract infra"

Option of base command 3: If user wants infrastructure data in input and a config file is mentioned to perform operations, you should detect that <config> file for operations which can be a random string and add them to the output like this:(default value for <config> file is "config.json")
"br-t9s.cli extract infra --config <config>"

Option of base command 3: If user wants the infrastructure data in input and a namespace is mentioned to perform operations, you should detect that <namespace> which can be a random string and add them to the output like this:(default value for <namespace> is "trirematics")
"br-t9s.cli extract infra --namespace <namespace>"

Option of base command 3: If user wants the infrastructure data in input and also want to debug the output, add argument "--verbose" besides them to the output like this:
"br-t9s.cli extract infra --verbose"



Base command 4: If the user wants the logs of a specific element, and no other options are mentioned, you should detect that <element> which can be random string, the output is like this:
"br-t9s.cli extract logs <element>"

Option of base command 4: If user wants logs of specific element in input and a config file is mentioned to perform operations, you should detect that <element> which can be random string and detect that <config> file for operations which can be a random string and add them to the output like this:(default value for <config> file is "config.json")
"br-t9s.cli extract logs <element> --config <config>"

Option of base command 4: If user wants logs of specific element in input and a namespace is mentioned to perform operations, you should detect that <element> which can be random string and detect that <namespace> for operations which can be a random string and add them to the output like this:(default value for <namespace> is "trirematics")
"br-t9s.cli extract logs <element> --namespace <namespace>"

Option of base command 4: If user wants logs of specific element in input and also wants to debug the output, you should detect that <element> which can be random string and add "--verbose" argument besides them to the output like this:
"br-t9s.cli extract logs <element> --verbose"

"""

def build_prompt(user_request: str) -> str:
    """Wrap the CLI specification and request in the original Mistral instruction format."""
    return f"<s>[INST] {COMMAND_SPECIFICATION} \n{user_request} \n[/INST]\n"

## 4. Fine-tuning data and historical training configuration

The training data is included in [`data/`](data/). Each JSONL record contains a `text` field with a Mistral `[INST] ... [/INST]` prompt and a target CLI command. There are **78 training, 10 validation, and 10 test** examples spanning `config`, `graph`, `infra`, and `logs` extraction.

The original development notebook printed a training invocation, but the command execution itself was commented out. The presence of the LoRA checkpoint does **not** establish a documented training log or measured held-out performance. The following command reproduces the *recorded configuration* (100 iterations, learning rate 1e-5, final 16 LoRA-adapted layers, validation every 10 steps), and writes to a **new** checkpoint rather than overwriting the supplied adapter:

```bash
python scripts/lora.py \
  --model mlx-community/Mistral-7B-Instruct-v0.2-4bit \
  --data data/ --train --iters 100 \
  --steps-per-eval 10 --val-batches -1 \
  --learning-rate 1e-5 --lora-layers 16 \
  --adapter-file adapters_retrained.npz --test
```

Running this command requires a compatible Apple Silicon MLX environment and the base model. No new training results are claimed here. The original `lora.py` evaluates token-level loss and perplexity when `--test` is provided; it does **not** compute CLI exact-match accuracy automatically.

## 5. Generate commands using the saved adapter

The helper below calls the included legacy `scripts/lora.py`, loads `adapters.npz`, and extracts a proposed command from the script's text output. It never executes `br-t9s.cli` commands. Do not treat generated commands as safe or valid without independent inspection.

The supplied legacy scripts are used intentionally because the standalone `.npz` adapter is not automatically compatible with newer MLX model-loading utilities.

In [ ]:
def generate_cli_command(user_request: str) -> str:
    """Generate one CLI command with the legacy MLX LoRA adapter."""
    if not LORA_SCRIPT.is_file():
        raise FileNotFoundError(
            f"Missing {LORA_SCRIPT}. Add the compatible legacy MLX LoRA scripts "
            "before running inference."
        )
    if not ADAPTER_PATH.is_file():
        raise FileNotFoundError(f"Missing adapter weights: {ADAPTER_PATH}")

    completed = subprocess.run(
        [
            sys.executable, str(LORA_SCRIPT),
            "--model", MODEL_PATH,
            "--adapter-file", str(ADAPTER_PATH),
            "--max-tokens", str(MAX_TOKENS),
            "--prompt", build_prompt(user_request),
        ],
        check=True,
        capture_output=True,
        text=True,
    )
    # The historical script echoes the prompt before generation.
    generation = completed.stdout.rsplit("[/INST]", 1)[-1]
    matches = re.findall(r"(?m)^br-t9s\.cli\s+extract\s+[^\r\n]+$", generation)
    if not matches:
        raise ValueError("No CLI command found in model output. Inspect the raw script output.")
    return matches[-1].strip()

## 6. Example requests

These examples cover graph extraction, configuration extraction with a specified target directory, and configuration export to a YAML path. Run the cell below **only after** setting up the legacy LoRA script and checkpoint.

In [ ]:
example_requests = [
    "Can you help troubleshoot the network diagram?",
    'I am interested in obtaining the configuration information of "oai-amf.oai-cn-rz2zjpq8" and moving it to the "/shared_data/" directory.',
    "I want configuration of gnb and write it in yaml file in current directory",
]

for request in example_requests:
    print(f"Request: {request}")
    print(f"Generated: {generate_cli_command(request)}\n")

### Recorded outputs from the original notebook

The following commands were extracted verbatim from three saved generation logs. **They are historical examples, not the outputs of an execution of this cleaned notebook.** The third example demonstrates a possible ambiguity in mapping a user request to the CLI's expected path format.

| Request (abbreviated) | Recorded generated command |
| --- | --- |
| Troubleshoot the network diagram | `br-t9s.cli extract graph --verbose` |
| Extract the configuration of `oai-amf.oai-cn-rz2zjpq8` to `/shared_data/` | `br-t9s.cli extract config oai-amf.oai-cn-rz2zjpq8 /shared_data/` |
| Export `gnb` configuration as YAML | `br-t9s.cli extract config gnb yaml/config.yaml` |

These outputs demonstrate command formatting for individual requests; they do not constitute a quantitative evaluation of command accuracy or correctness. CLI compatibility has not been independently tested here.

## 7. Scope and limitations

- Only **natural-language-to-CLI command generation** is included. An experimental classifier was not trained and is excluded.
- The dataset contains **98 small, task-specific examples**. Many are paraphrases of familiar commands, and validation/test target commands also occur in training. These splits are not evidence of generalization to unseen command grammars.
- The recorded generated commands are illustrative, and the notebook contains **no quantitative held-out command-accuracy benchmark**.
- CLI syntax and generated arguments have not been verified against a live management system. **No generated command is automatically executed.**
- Training and inference have not been rerun here using Apple Silicon. Compatibility with modern MLX releases remains to be validated.

### Attribution

The MLX fine-tuning and model-loading scripts retain their original Apple copyright notices and derive from the open-source [Apple MLX examples](https://github.com/ml-explore/mlx-examples). The starting notebook also drew on a tutorial by [Shaw Talebi](https://youtu.be/3PIqhdRzhxE). See [`THIRD_PARTY_NOTICES.md`](THIRD_PARTY_NOTICES.md).